# Notebook 4: QGAN — Quantum Generative Adversarial Network

Train a quantum generator to learn a target probability distribution.

The QGAN has:
- **Generator**: A parameterized quantum circuit that outputs a probability distribution
- **Discriminator**: A classical neural network that distinguishes real vs generated samples

We train the generator to reproduce a log-normal distribution.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from qforge.algo.qgan import QGAN

## Target Distribution

Generate samples from a log-normal distribution (common in financial modeling).

In [ ]:
np.random.seed(42)

# Log-normal target
real_data = np.random.lognormal(mean=1.0, sigma=0.5, size=5000)
# Clip to keep within reasonable range
real_data = real_data[real_data < 10]

plt.figure(figsize=(8, 4))
plt.hist(real_data, bins=50, density=True, alpha=0.7, label='Target (log-normal)')
plt.xlabel('Value')
plt.ylabel('Density')
plt.title('Target Distribution')
plt.legend()
plt.show()

print(f"Samples: {len(real_data)}, Mean: {real_data.mean():.2f}, Std: {real_data.std():.2f}")

## Train the QGAN

With 3 qubits, the generator outputs a distribution over $2^3 = 8$ bins.

In [ ]:
qgan = QGAN(n_qubits=3, n_layers=4)

g_history, d_history = qgan.train(
    real_data=real_data,
    steps=150,
    gen_lr=0.05,
    disc_lr=0.01,
    disc_steps=3
)

print(f"Final generator loss: {g_history[-1]:.4f}")
print(f"Final discriminator loss: {d_history[-1]:.4f}")

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

ax1.plot(g_history, label='Generator', linewidth=2)
ax1.plot(d_history, label='Discriminator', linewidth=2)
ax1.set_xlabel('Step')
ax1.set_ylabel('Loss')
ax1.set_title('QGAN Training')
ax1.legend()
ax1.grid(True, alpha=0.3)

# Compare distributions
gen_dist = qgan.generated_distribution()
n_bins = len(gen_dist)
real_hist, bin_edges = np.histogram(real_data, bins=n_bins, density=True)
real_hist = real_hist / real_hist.sum()  # normalize to probability

x = np.arange(n_bins)
width = 0.35
ax2.bar(x - width/2, real_hist, width, label='Target', alpha=0.7)
ax2.bar(x + width/2, gen_dist, width, label='Generated', alpha=0.7)
ax2.set_xlabel('Bin')
ax2.set_ylabel('Probability')
ax2.set_title('Target vs Generated Distribution')
ax2.legend()

plt.tight_layout()
plt.show()

In [ ]:
# Quality metric
kl = qgan.kl_divergence(real_data)
print(f"KL divergence: {kl:.4f}")
print("(lower is better, 0 = perfect match)")

## Sampling from the Generator

Once trained, we can draw new samples from the quantum generator.

In [ ]:
# Draw 2000 samples from the trained generator
generated_samples = qgan.sample(2000)

plt.figure(figsize=(8, 4))
plt.hist(real_data, bins=30, density=True, alpha=0.5, label='Target data')
plt.hist(generated_samples, bins=n_bins, density=True, alpha=0.5, label='QGAN samples')
plt.xlabel('Value')
plt.ylabel('Density')
plt.title('QGAN Samples vs Target Data')
plt.legend()
plt.show()

## Training with Different Qubit Counts

More qubits = more bins = finer resolution.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

for idx, nq in enumerate([2, 3, 4]):
    qgan_i = QGAN(n_qubits=nq, n_layers=4)
    qgan_i.train(real_data, steps=100, gen_lr=0.05, disc_lr=0.01)

    gen_dist = qgan_i.generated_distribution()
    n_bins_i = len(gen_dist)
    real_hist, _ = np.histogram(real_data, bins=n_bins_i, density=True)
    real_hist = real_hist / real_hist.sum()

    x = np.arange(n_bins_i)
    axes[idx].bar(x - 0.2, real_hist, 0.4, label='Target', alpha=0.7)
    axes[idx].bar(x + 0.2, gen_dist, 0.4, label='Generated', alpha=0.7)
    kl = qgan_i.kl_divergence(real_data)
    axes[idx].set_title(f'{nq} qubits ({2**nq} bins), KL={kl:.3f}')
    axes[idx].legend(fontsize=8)
    axes[idx].set_xlabel('Bin')
    axes[idx].set_ylabel('Probability')

plt.suptitle('QGAN Resolution vs Qubit Count', fontsize=14)
plt.tight_layout()
plt.show()